In [4]:
from urllib.parse import parse_qs, quote_plus, urlparse

from bs4 import BeautifulSoup
from pydantic import BaseModel, ValidationError
from pymongo import MongoClient



In [10]:
MONGO_URI="mongodb://localhost:27017/"
DB_NAME = "kikihi"
COLLECTION_NAME = "keyboard"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]
collection = db[COLLECTION_NAME]

In [ ]:
# remove_words = ['교체용스위치', '기계식', '스위치', '독자 규격']
# keywords = [
#     # 스위치 유형 및 특성
#     "적축", "청축", "갈축", "흑축", "백축", "무접점", "광축", "토프레", "광학 스위치",

#     # 키보드 구조 및 형상
#     "텐키리스", "풀사이즈", "75%", "65%", "미니", "분리형", "스플릿 키보드",

#     # 키캡 및 레이아웃
#     "PBT", "ABS", "사출", "이중사출", "열보강", "스태빌라이저", "스탭스컬쳐",

#     # 타이핑 감각 및 촉감
#     "클릭감", "리니어", "넌클릭", "탭핑", "피드백", "반발력", "무소음",

#     # 기타 기능 및 특징
#     "RGB 백라이트", "무한 동시입력", "무접점 방진", "방수", "블루투스", "유선", "무선",
#     "핫스왑", "내장 메모리", "매크로", "프로그램 가능", "LED",

#     # 기타 용어
#     "스테빌", "포텐셔미터", "스위치 핀", "스프링", "하우징", "키스위치", "키프레스"
    
#     # 하우징
#     "Tray-Mount", "Top-Mount", "Gasket-Mount",
#     "플라스틱", "알루미늄", "아크릴",
#     "60%", "TKL", "풀배열",
#     "MX 호환", "Alps 호환", "Topre 호환",
#     "투명", "반투명",

#     # 키캡
#     "MX 스템", "Alps 스템", "Topre 스템",
#     "SA", "OEM", "Cherry", "DSA",
#     "ABS", "PBT", "PC",
#     "ANSI", "ISO", "JIS",
#     "두꺼운 키캡", "얇은 키캡",
#     "레이저 각인", "더블샷", "승화 인쇄",
#     "material", "keycapSet",

#     # 스위치
#     "3핀", "5핀",
#     "45g", "60g", "80g",
#     "MX 스타일", "Alps 스타일",
#     "RGB 호환", "SMD LED 호환",
#     "윤활 처리", "비윤활",
#     "저소음", "클릭형", "리니어", "택타일",
#     "체리식 호환", "코스타식 호환"
# ]


keywords_all = [
    # 스위치 유형 및 특성
    "적축", "red switch", "청축", "blue switch", "갈축", "brown switch",
    "흑축", "black switch", "백축", "white switch",
    "무접점", "electrostatic", "광축", "optical switch", "토프레", "topre",
    "광학 스위치", "optical",

    # 키보드 구조 및 형상
    "텐키리스", "tenkeyless", "tkl", "풀사이즈", "full size",
    "75%", "65%", "60%", "미니", "mini",
    "분리형", "split", "스플릿 키보드", "split keyboard", "풀배열", "full layout",

    # 키캡 및 레이아웃
    "PBT", "pbt", "ABS", "abs", "PC", "pc",
    "사출", "injection", "이중사출", "double shot", "더블샷",
    "열보강", "heat treated",
    "스태빌라이저", "stabilizer", "스테빌", "stab", "스탭스컬쳐", "step sculpture",
    "MX 스템", "mx stem", "Alps 스템", "alps stem", "Topre 스템", "topre stem",

    # 타이핑 감각 및 촉감
    "클릭감", "clicky", "리니어", "linear", "넌클릭", "non-clicky",
    "탭핑", "tapping", "피드백", "feedback", "반발력", "rebound force",
    "무소음", "silent", "저소음", "low noise", "클릭형", "click type", "택타일", "tactile",

    # 기능 및 특성
    "RGB 백라이트", "rgb backlight", "RGB", "rgb", "LED", "led",
    "무한 동시입력", "n-key rollover", "nkey rollover",
    "무접점 방진", "dustproof", "방수", "waterproof",
    "블루투스", "bluetooth", "유선", "wired", "무선", "wireless",
    "핫스왑", "hot swap", "hotswap", "내장 메모리", "internal memory",
    "매크로", "macro", "프로그램 가능", "programmable",

    # 기타 용어 및 부품
    "포텐셔미터", "potentiometer", "스위치 핀", "switch pin",
    "스프링", "spring", "하우징", "housing", "키스위치", "key switch",
    "키프레스", "key press", "stem",

    # 하우징
    "Tray-Mount", "tray mount", "Top-Mount", "top mount", "Gasket-Mount", "gasket mount",
    "플라스틱", "plastic", "알루미늄", "aluminum", "아크릴", "acrylic",
    "투명", "transparent", "반투명", "translucent",
    "MX 호환", "mx compatible", "Alps 호환", "alps compatible", "Topre 호환", "topre compatible",
    "체리식 호환", "cherry compatible", "코스타식 호환", "costar compatible",

    # 키캡 프로파일 & 인쇄
    "SA", "sa", "OEM", "oem", "Cherry", "cherry", "DSA", "dsa",
    "ANSI", "ansi", "ISO", "iso", "JIS", "jis",
    "두꺼운 키캡", "thick keycap", "얇은 키캡", "thin keycap",
    "레이저 각인", "laser etched", "승화 인쇄", "dye sublimation",
    "material", "keycapSet", "keycap set",

    # 스위치 상세 스펙
    # 구조 / 핀 수
    "3핀", "3 pin", "5핀", "5 pin", "3핀 핫스왑", "5핀 납땜",
    
    # 키압
    "35g", "45g", "50g", "55g", "60g", "62g", "65g", "67g", "68g", "70g", "78g", "80g",
    
    # 스위치 종류 / 브랜드
    "MX", "mx", "Alps", "alps", "Topre", "로터리", "광축", "정전용량", "기계식", "무접점",
    
    # 특성
    "윤활", "비윤활", "lubed", "unlubed", "스무스", "부드러움", "걸림없음",
    "찰칵", "클릭", "리니어", "넌클릭", "타이핑감", "반발력", "정숙성",
    
    # LED / 조명
    "RGB", "rgb", "LED", "백라이트", "투명하우징",
    
    # 기타
    "듀얼스프링", "로프레스", "고압", "저압", "정방향", "역방향",
    "하우징", "스템", "금도금", "폴링레이트"

]



## 쓸데 없는 단어 제거

In [13]:
def clean_description(desc, remove_words):
    if isinstance(desc, list):
        desc = " ".join(map(str, desc))
    for w in remove_words:
        desc = desc.replace(w, "")
    return desc.strip()


## 

In [14]:
import re

def extract_keywords_from_fields(name, spec_table, keywords):
    pcs_patterns = [
        re.compile(r'(\d+\+?\d*)\s*개[ ,]*(.*?)(?=,|$)'),   # ex: "3개, 모터"
        re.compile(r'(\d+)\s*(pcs|PCS|pcs)')                # ex: "2 pcs"
    ]
    axis_pattern = re.compile(r'(\d+)\s*축')                # ex: "3축"
    pin_pattern = re.compile(r'(\d+)\s*핀')                 # ex: "4핀"
    named_axis_pattern = re.compile(r'(\b\w{1,10}|[가-힣]{1,10})\s*축')  # ex: "X축", "로봇축"

    field_str = f"{name or ''} {spec_table or ''}"
    found_words = {kw for kw in keywords if kw in field_str}
    desc_list = []

    # 개수 + 설명 or pcs 패턴
    for pattern in pcs_patterns:
        for m in pattern.finditer(field_str):
            if pattern == pcs_patterns[0]:
                qty, detail = m.groups()
                entry = f"{qty}개"
                if detail.strip():
                    entry += f", {detail.strip()}"
                desc_list.append(entry)
            # else:
            #     qty, _pcs = m.groups()
            #     desc_list.append(f"{qty}PCS")

    # 숫자 + 축/핀
    desc_list += [f"{m.group(1)}축" for m in axis_pattern.finditer(field_str)]
    desc_list += [f"{m.group(1)}핀" for m in pin_pattern.finditer(field_str)]

    # 문자 기반 축명
    for m in named_axis_pattern.finditer(field_str):
        axis = m.group(1).strip()
        if axis and f"{axis}축" not in desc_list:
            desc_list.append(axis+"축")

    # 키워드 포함
    desc_list.extend(sorted(found_words))

    # 중복 제거 (순서 보존)
    return list(dict.fromkeys(filter(None, desc_list)))


## 제거

In [ ]:
def clean_description(desc_list, remove_words):
    cleaned = []
    for item in desc_list:
        # 문자열인지 확인
        if not isinstance(item, str):
            continue
        
        # 1. 불용어 제거
        for word in remove_words:
            item = item.replace(word, "")
        
        # 2. 특수문자 제거 (',', "'", '"')
        item = item.replace(",", "")  # 쉼표 제거
        item = item.replace("'", "").replace('"', "")
        
        # 3. 양 끝 공백 제거
        item = item.strip()
        
        # 4. 빈 문자열은 제거
        if item:
            cleaned.append(item)
    return cleaned
import re



In [35]:
raw_data = list(collection.find())

for item in raw_data:
    if "spec_table" in item and isinstance(item["spec_table"], dict):
        item["spec_table"] = clean_spec_table(item["spec_table"])

# 결과 확인 (앞 5개만)
for r in raw_data[:5]:
    print(r.get("spec_table", ""))


{'제조회사': 'MOUNTAIN', '등록년월': '2022년 10월', '개수': '110개', '키압': '55g', 'KC인증 > 적합성평가인증': '상세설명 / 판매 사이트 문의인증번호 확인', 'KC인증 > 안전확인인증': '상세설명 / 판매 사이트 문의인증번호 확인'}
{'제조회사': '앱코 HACKER', '등록년월': '2017년 06월', '개수': '8개', '키보드구조 > 스위치 교체형': '○', 'KC인증 > 적합성평가인증': '상세설명 / 판매 사이트 문의인증번호 확인', 'KC인증 > 안전확인인증': '상세설명 / 판매 사이트 문의인증번호 확인'}
{'제조회사': 'Razer', '등록년월': '2023년 10월', '개수': '36개', '키압': '50g', '제품 보증 > A/S 보증기간': '1년 보증', 'KC인증 > 적합성평가인증': '상세설명 / 판매 사이트 문의인증번호 확인', 'KC인증 > 안전확인인증': '상세설명 / 판매 사이트 문의인증번호 확인'}
{'제조회사': '글로리어스', '등록년월': '2021년 09월', '개수': '36개', '키압': '60g', 'KC인증 > 적합성평가인증': '상세설명 / 판매 사이트 문의인증번호 확인', 'KC인증 > 안전확인인증': '상세설명 / 판매 사이트 문의인증번호 확인'}
{'제조회사': '글로리어스', '등록년월': '2021년 08월', '개수': '36개', '키압': '60g', 'KC인증 > 적합성평가인증': '상세설명 / 판매 사이트 문의인증번호 확인', 'KC인증 > 안전확인인증': '상세설명 / 판매 사이트 문의인증번호 확인'}


## 제거 함수 적용

In [32]:
# MongoDB에서 데이터 불러오기
raw_data = list(collection.find())

# 제거할 단어 리스트가 있다면
remove_words = ["기계식","스위치"]

# description 정제
for item in raw_data:
    if "description" in item and isinstance(item["description"], list):
        item["description"] = clean_description(item["description"], remove_words)

# 정제 결과 출력 (앞 100개)
for r in raw_data[:100]:
    print(r.get("description", ""))


['110개', '55g']
['8개']
['36개', '50g']
['36개', '60g', '윤활']
['36개', '60g']
['36개', '67g']
['120개', '45g', '윤활']
['36개', '자석축', '45g', '윤활']
['36개', '67g', '윤활']
['36개', '55g', 'pc', '윤활']
['36개', '자석축', '45g', '윤활']
['36개', '자석축', '45g', '윤활']
['36개', '자석축', '윤활']
['36개', '자석축', '55g', '윤활']
['1개', '리니어']
['110개', '50g']
['110개', '45g']
['105개', '35g', '저소음', '택타일']
['110개', '50g', '리니어']
['35개', '리니어']
['55개', '특주축', '윤활']
['35개', '리니어', '저소음']
['45개', '55g']
['35개', 'MX', '리니어']
['110개', '45g']
['110개', '55g']
['35개', '50g', 'Cherry', 'MX', 'RGB']
['35개', '리니어']
['35개', 'MX', '리니어']
['110개', '45g', 'MX']
['110개', 'pc', '저소음']
['110개', '50g']
['35개', '50g', '리니어']
['110개']
['45개', 'pc']
['100개', '60g']
['35개', '45g', '저소음']
['45개', '50g', 'pc']
['35개', '45g', 'MX', '리니어']
['105개', '55g', '리니어', '저소음']
['45개', 'pc']
['90개']
['35개', '45g', '리니어']
['35개', '50g', 'Cherry', 'MX']
['90개', '레몬축', '55g', '택타일']
['35개', '저소음']
['1개', '60g', '리니어']
['35개', '리니어']
['105개', '45g', '리니어']
['1개', '리

## 추가 함수 적용

In [17]:
# 1. MongoDB에서 모든 문서 읽기
raw_data = list(collection.find())

# 2. 각 문서 description 정제
for item in raw_data:
    name = item.get("name", "")
    spec_table = item.get("spec_table", "")
    desc_list = extract_keywords_from_fields(name, spec_table, keywords_all)
    item["description"] = desc_list
    
# 3. 정제된 raw_data 출력 예시 (첫 5개만)
for r in raw_data[:300]:
    print(r["description"])  # description 필드만 출력

['4개, 입(Set)']
['4개, 입(Set)']
['1개, 입']
['1개, 입']
['기계식', '아크릴']
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
['더블샷']
['기계식']
['PBT']
['PBT']
['PBT']
[]
[]
[]
[]
[]
[]
[]
['기계식']
['PBT', '반투명', '사출', '투명']
['기계식']
[]
['PBT', '반투명', '사출', '투명']
['OEM', 'PBT', '기계식']
['PBT', '기계식']
['PBT', '기계식']
['OEM', 'PBT']
['PBT', '기계식']
['MX', 'PBT', '기계식']
['기계식']
['PBT', 'pbt']
['PBT']
['PBT']
['PBT', 'pbt', '기계식']
['PBT', 'pbt', '기계식']
["138개, '", 'PBT']
["132개, '", 'PBT']
["2개, '", 'PBT', 'SA', 'pc', '더블샷']
["132개, '", 'PBT']
["132개, '", 'PBT']
["139개, '", 'PBT']
["139개, '", 'PBT']
["139개, '", 'PBT']
["139개, '", 'PBT']
["173개, '", 'PBT']
["7개, '"]
["165개, '", 'PBT', 'PC', '투명']
["139개, '", 'PBT']
["139개, '", 'PBT']
["173개, '", 'PBT']
['PBT']
["165개, '", 'PBT', 'PC', '투명']
[]
['3개, 세트', '기계식', '아크릴']
['반투명', '투명']
[]
[]
["139개, '", 'PBT']
["165개, '", 'PBT', 'PC', '투명']
['기계식']
['기계식']
[]
['기계식']
['기계식']
[]
['기계식']
['기계식']
[]
['Cherry', 'DSA', 'MX', 'PBT', 'SA', '기계식', '미니']
['OEM', '반투명', '투명

# keyboard 전처리 
- name에서 제조사 빼서 manufactur로 넣기
- options을 desciption으로 복사하기
- description에서 쓸데 없는 부분 삭제
- spec_table에서 쓸데 없는 부분 삭제(제조사 웹사이트 바로 가기 삭제)

In [11]:
description_remove_words = ['1ms 응답속도','키보드','한/영 정각','착탈식 케이블','교체용스위치', '기계식', '스위치', '독자 규격']

spec_table_remove_words = ['KC인증 > 적합성평가인증','KC인증 > 안전확인인증','제조사 웹사이트 바로 가기', '제조사 웹사이트 바로가기', '제조사 웹사이트', '제조사 홈페이지', '제조사 사이트']


In [ ]:
raw_data = list(collection.find())

def extract_manufacturer(product_title):
    parts = product_title.split(' ', 1)
    manufacturer = parts[0]
    product_name = parts[1] if len(parts) > 1 else ''
    return manufacturer, product_name

# 모든 문서를 대상으로 업데이트 수행
for doc in raw_data:
    name = doc.get("name", "")
    if name:
        manufacturer, new_name = extract_manufacturer(name)
        doc["manufacturer"] = manufacturer
        doc["name"] = new_name


for r in raw_data[:300]:
    print(r.get("name", ""))
    print(r.get("manufacturer", ""))



In [15]:
# 4. DB 업데이트 - raw_data 리스트 기준으로 업데이트
for item in raw_data:
    _id = item["_id"]
    manufacturer = item.get("manufacturer", "")
    new_name = item.get("name", "")
    collection.update_one(
        {"_id": _id},
        {
            "$set": {
                "manufacturer": manufacturer,
                "name": new_name
            }
        }
    )


In [18]:
import re
from typing import Dict, Any, List


## 2. options을 desciption으로 복사하기
def copy_options_to_description(item: dict) -> dict:
    """
    options 리스트를 description 필드로 복사
    기존 description이 리스트면 병합
    options가 없거나 빈 리스트면 처리하지 않고 continue(호출문 맥락에서 반복문 내 사용시)
    """
    options = item.get('options', [])
    if not options:
        # options가 없거나 빈 리스트면 아무 작업도 하지 않고 함수 종료
        return item

    if isinstance(item.get('description'), list):
        item['description'] = options + item['description']

    return item
raw_data = list(collection.find())


for item in raw_data:
    result = copy_options_to_description(item)
    if result is None:
        # description이 리스트가 아니어서 처리를 못한 경우(else문)
        print("description이 리스트가 아니라서 처리 건너뜀:", item)
        continue
    print("처리된 아이템:", result.get("description", ""))


처리된 아이템: ['딥 사운드축', '조약돌 슈팅스타축', '중고', '키보드', '풀배열', '유선+무선', '기계식', '104키', '전용동글(리시버)', '블루투스', '4000mAh', '1ms 응답속도', 'RGB 백라이트', '스텝스컬쳐2', '스테빌라이저', '흡음재', '다이얼(노브)', 'LCD', 'PBT', '염료승화 방식', '한/영 정각', '착탈식 케이블']
처리된 아이템: ['저소음 바다축', '저소음 솜사탕축', '경해축', '세이야축', '황축', '키보드', '풀배열', '유선+무선', '기계식', '108키', '전용동글(리시버)', '블루투스', '8000mAh', '1ms 응답속도', '매크로키', 'RGB 백라이트', '스텝스컬쳐2', '스테빌라이저', '스위치 교체형', '흡음재', 'PBT', '이중사출 키캡', '한/영 정각', '추가키캡', '4개', '착탈식 케이블']
처리된 아이템: ['적축', '갈축', '청축', '중고', '키보드', '풀배열', '유선', '기계식', '104키', '스위치', 'JIXIAN', '1ms 응답속도', '비키스타일', '레인보우 백라이트', '스텝스컬쳐2', '스테빌라이저', '스위치 교체형', '이중사출 키캡']
처리된 아이템: ['저소음 바다축', '세이야축', '중고', '키보드', '텐키리스', '유선', '기계식', '87키', '1ms 응답속도', '숫자키없음', 'RGB 백라이트', '스텝스컬쳐2', '스테빌라이저', '스위치 교체형', '흡음재', 'PBT', '이중사출 키캡', '한/영 정각', '추가키캡', '7개', '착탈식 케이블']
처리된 아이템: ['적축', '갈축', '청축', '키보드', '풀배열', '유선', '기계식', '104키', '스위치', '1ms 응답속도', '비키스타일', '레인보우 백라이트', '스텝스컬쳐2', '스테빌라이저', '스위치 교체형', '방진기능', 'ABS', '이중사출 키캡', '한/영 정각', '직조(패브릭) 

In [19]:


def copy_options_to_description(item: dict) -> dict:
    """
    options 리스트를 description 필드로 복사
    기존 description이 리스트면 병합
    options가 없거나 빈 리스트면 아무 작업도 하지 않고 item 그대로 반환
    """
    options = item.get('options', [])
    if not options:
        # options가 없거나 빈 리스트면 아무 작업 안함
        return item

    if isinstance(item.get('description'), list):
        item['description'] = options + item['description']
    # description이 리스트가 아니면 변형하지 않고 그냥 반환
    return item

# MongoDB에서 데이터 가져오기
raw_data = list(collection.find())

for item in raw_data:
    original_description = item.get("description")  # 변경 전 description 저장 (확인용)
    
    # description 필드 수정 함수 호출
    updated_item = copy_options_to_description(item)
    
    # description이 리스트가 아니면 처리 안하고 건너뛰고 싶으면 이렇게 (필요에 따라)
    if not isinstance(updated_item.get('description'), list):
        print("description이 리스트가 아니라서 처리 건너뜀:", item)
        continue
    
    # 변경된 description 필드를 DB에 업데이트 (같은 _id 사용)
    collection.update_one(
        {"_id": item["_id"]},
        {"$set": {"description": updated_item["description"]}}
    )
    print(f"업데이트 완료: _id={item['_id']}")
    print(f"기존 description: {original_description}")
    print(f"변경된 description: {updated_item['description']}\n")

print("모든 문서 업데이트 완료!")


업데이트 완료: _id=688701674f949c449d02073a
기존 description: ['키보드', '풀배열', '유선+무선', '기계식', '104키', '전용동글(리시버)', '블루투스', '4000mAh', '1ms 응답속도', 'RGB 백라이트', '스텝스컬쳐2', '스테빌라이저', '흡음재', '다이얼(노브)', 'LCD', 'PBT', '염료승화 방식', '한/영 정각', '착탈식 케이블']
변경된 description: ['딥 사운드축', '조약돌 슈팅스타축', '중고', '키보드', '풀배열', '유선+무선', '기계식', '104키', '전용동글(리시버)', '블루투스', '4000mAh', '1ms 응답속도', 'RGB 백라이트', '스텝스컬쳐2', '스테빌라이저', '흡음재', '다이얼(노브)', 'LCD', 'PBT', '염료승화 방식', '한/영 정각', '착탈식 케이블']

업데이트 완료: _id=688701764f949c449d02073b
기존 description: ['키보드', '풀배열', '유선+무선', '기계식', '108키', '전용동글(리시버)', '블루투스', '8000mAh', '1ms 응답속도', '매크로키', 'RGB 백라이트', '스텝스컬쳐2', '스테빌라이저', '스위치 교체형', '흡음재', 'PBT', '이중사출 키캡', '한/영 정각', '추가키캡', '4개', '착탈식 케이블']
변경된 description: ['저소음 바다축', '저소음 솜사탕축', '경해축', '세이야축', '황축', '키보드', '풀배열', '유선+무선', '기계식', '108키', '전용동글(리시버)', '블루투스', '8000mAh', '1ms 응답속도', '매크로키', 'RGB 백라이트', '스텝스컬쳐2', '스테빌라이저', '스위치 교체형', '흡음재', 'PBT', '이중사출 키캡', '한/영 정각', '추가키캡', '4개', '착탈식 케이블']

업데이트 완료: _id=688701854f949c449d02073c

In [23]:
## 3. description에서 쓸데 없는 부분 삭제

# 삭제할 KC인증 관련 키 리스트
keys_to_remove = ["KC인증 > 적합성평가인증", "KC인증 > 안전확인인증"]

# DB 문서 전체 가져오기
raw_data = list(collection.find())

for doc in raw_data:
    spec_table = doc.get("spec_table", None)
    if not isinstance(spec_table, dict):
        # spec_table 필드가 없거나 dict가 아니면 건너뜀
        continue

    # 1. 삭제할 키 제거
    for key in keys_to_remove:
        spec_table.pop(key, None)

    # 2. '제조회사' 값에서 '(제조사 웹사이트 바로가기)' 제거
    if "제조회사" in spec_table:
        spec_table["제조회사"] = spec_table["제조회사"].replace("(제조사 웹사이트 바로가기)", "").strip()

    # 3. 변경된 spec_table을 DB에 업데이트
    collection.update_one(
        {"_id": doc["_id"]},
        {"$set": {"spec_table": spec_table}}
    )

    print(f"문서 _id={doc['spec_table']} spec_table 업데이트 완료")

print("모든 문서 spec_table 필드 수정 및 DB 반영 완료!")


문서 _id={'제조회사': '앱코', '등록년월': '2024년 11월', '사이즈': '풀배열', '연결 방식': '유선+무선', '접점 방식': '기계식', '키 배열': '104키', '배터리': '내장 배터리', '배터리 용량': '4000mAh', '인터페이스': 'USB', '용도 > 사무용': '○', '기능 > 키 스위치': '딥 사운드 축', '기능 > 키압': '45g', '기능 > 매크로 기능': 'S/W매크로', '기능 > 동시입력': '무한', '기능 > 응답속도': '1ms 응답속도', '키보드구조 > RGB 백라이트': '○', '키보드구조 > 스텝스컬쳐2': '○', '키보드구조 > 스테빌라이저': '○', '키보드구조 > 흡음재': '○', '키보드구조 > 다이얼(노브)': '○', '키보드구조 > LCD': '○', '키보드구조 > C타입 포트': '○', '키캡 > 키캡 재질': 'PBT', '키캡 > 키캡 각인방식': '염료승화 방식', '키캡 > 각인 위치': '한/영 정각', '부가 기능 > 멀티페어링': '○', '부가 기능 > 멀티미디어': '○', '케이블 > 착탈식 케이블': '○', '추가 구성 > 루프': '○', '추가 구성 > 키캡 리무버': '○', '크기(가로x세로x높이) > 가로': '438.47mm', '크기(가로x세로x높이) > 세로': '138.46mm', '크기(가로x세로x높이) > 높이': '43.09mm', '크기(가로x세로x높이) > 무게': '1163g', '제품 보증 > A/S 보증기간': '1년 보증'} spec_table 업데이트 완료
문서 _id={'제조회사': 'AULA', '등록년월': '2024년 10월', '사이즈': '풀배열', '연결 방식': '유선+무선', '접점 방식': '기계식', '키 배열': '108키', '블루투스 버전': '5.0', '배터리': '내장 배터리', '배터리 용량': '8000mAh', '인터페이스': 'USB', '용도 > 사무용': '○'

In [ ]:

## 4. desciprion에서 쓸데 없는 부분 삭제
# 삭제할 단어 리스트
description_remove_words = [    '키보드','1ms 응답속도','한/영 정각','착탈식 케이블','교체용스위치', '기계식', '스위치', '독자 규격'
]



## 저장

In [ ]:
# 4. DB 업데이트 - raw_data 리스트 기준으로 업데이트
for item in raw_data:   # result 대신 raw_data 사용!
    _id = item["_id"]
    collection.update_one(
        {"_id": _id},
        {"$set": {"manufact": item["spec_table"]}}
    )

In [ ]:
# # 4. DB 업데이트 - raw_data 리스트 기준으로 업데이트
# for item in raw_data:   # result 대신 raw_data 사용!
#     _id = item["_id"]
#     collection.update_one(
#         {"_id": _id},
#         {"$set": {"description": item["description"]}}
#     )

## 키보드 전처리
- name앞에 제조사 떼어내서 manufactur에 넣기

## 하우징

In [15]:
from pymongo import MongoClient
import re
from collections import Counter

client = MongoClient("mongodb://localhost:27017")
db = client['kikihi']
collection = db['housing']

keywords = [
    "ABS", "PBT", "아크릴", "Acrylic", "알루미늄", "Aluminum",
    "폴리카보네이트", "Polycarbonate", "나무", "Wood", "원목",
    "스테인리스", "Stainless", "티타늄", "Titanium", "FR4",
    "브라스", "Brass", "황동", "플라스틱", "Plastic", "카본", "Carbon",
    "유리", "Glass", "POM", "폼"
]

word_count = Counter()

documents = collection.find({}, {"description": 1})

for doc in documents:
    desc = doc.get("description") or doc.get("name") or doc.get("spec_table")
    if isinstance(desc, list):
        # 리스트 내 모든 요소를 문자열로 변환 후 합치기 (공백으로 구분)
        desc = " ".join(str(item) for item in desc)
    elif not isinstance(desc, str):
        desc = ""
    for kw in keywords:
        word_count[kw] += len(re.findall(kw, desc, re.IGNORECASE))


for kw, cnt in word_count.items():
    print(f"{kw}: {cnt}회")


ABS: 2회
PBT: 0회
아크릴: 0회
Acrylic: 0회
알루미늄: 22회
Aluminum: 0회
폴리카보네이트: 0회
Polycarbonate: 0회
나무: 0회
Wood: 0회
원목: 0회
스테인리스: 0회
Stainless: 0회
티타늄: 0회
Titanium: 0회
FR4: 0회
브라스: 0회
Brass: 0회
황동: 0회
플라스틱: 1회
Plastic: 0회
카본: 0회
Carbon: 0회
유리: 0회
Glass: 0회
POM: 0회
폼: 0회


## 축

In [24]:
from pymongo import MongoClient
import re
from collections import Counter

client = MongoClient("mongodb://localhost:27017")
db = client['kikihi']  # 본인 DB명에 맞게 변경
collection = db['switch']

word_count = Counter()

# '축'으로 끝나는 한글 단어 추출 정규식 (예: 청축, 갈축, 적축 등)
pattern = re.compile(r"([가-힣]+축)\b")

# description, options 필드가 문자열 또는 리스트일 수 있으므로 처리 함수
def get_text_from_field(field_value):
    if isinstance(field_value, str):
        return field_value
    elif isinstance(field_value, list):
        return " ".join(str(x) for x in field_value)
    else:
        return ""

documents = collection.find({}, {"description": 1, "options": 1})

for doc in documents:
    desc_text = get_text_from_field(doc.get("description"))
    options_text = get_text_from_field(doc.get("options"))

    combined_text = f"{desc_text} {options_text}"

    matches = pattern.findall(combined_text)
    for m in matches:
        word_count[m] += 1

# 결과 출력
for word, cnt in word_count.most_common():
    print(f"{word}: {cnt}회")


적축: 33회
갈축: 29회
흑축: 16회
청축: 12회
자석축: 8회
백축: 7회
바나나축: 5회
은축: 5회
황축: 5회
키보드축: 4회
체리축: 4회
윤활축: 4회
광축: 3회
민트축: 2회
교체축: 2회
공주축: 2회
카일박스축: 2회
핑크축: 2회
특주축: 1회
화이트축: 1회
오렌지축: 1회
레몬축: 1회
라즈베리축: 1회
카라멜라떼축: 1회
건반축: 1회
강백축: 1회
세이야축: 1회
구름축: 1회
고래축: 1회
크림축: 1회
연꽃축: 1회
극지여우축: 1회
제이드축: 1회
피치축: 1회
밀감축: 1회
스노우축: 1회
라임축: 1회
바다축: 1회
녹축: 1회
저소음머스타드축: 1회
아이스축: 1회
람보축: 1회


In [25]:
from pymongo import MongoClient
import re
from collections import Counter

client = MongoClient("mongodb://localhost:27017")
db = client['kikihi']  # 본인 DB명에 맞게 변경
collection = db['switch']

word_count = Counter()

# '축'으로 끝나는 한글 단어 추출 정규식 (예: 청축, 갈축, 적축 등)
# pattern = re.compile(r"([가-힣]+축)\b")
pattern=['리니어','택타일']
# description, options 필드가 문자열 또는 리스트일 수 있으므로 처리 함수
def get_text_from_field(field_value):
    if isinstance(field_value, str):
        return field_value
    elif isinstance(field_value, list):
        return " ".join(str(x) for x in field_value)
    else:
        return ""

documents = collection.find({}, {"description": 1, "options": 1})

for doc in documents:
    desc_text = get_text_from_field(doc.get("description"))
    options_text = get_text_from_field(doc.get("options"))

    combined_text = f"{desc_text} {options_text}"

    matches = pattern.findall(combined_text)
    for m in matches:
        word_count[m] += 1

# 결과 출력
for word, cnt in word_count.most_common():
    print(f"{word}: {cnt}회")


AttributeError: 'list' object has no attribute 'findall'

## 키캡

In [11]:
from pymongo import MongoClient
import re
from collections import Counter

client = MongoClient("mongodb://localhost:27017")
db = client['kikihi']  # 본인 DB명에 맞게 변경
collection = db['keycap']

# 키캡 프로필명 리스트
keywords = ["OEM", "Cherry", "SA", "DSA", "XDA", "KAT", "KAM", "MT3"]

word_count = Counter()

# 여러 필드에 대해 문자열/리스트 처리 함수
def get_text_from_field(field_value):
    if isinstance(field_value, str):
        return field_value
    elif isinstance(field_value, list):
        return " ".join(str(x) for x in field_value)
    else:
        return ""

# MongoDB 문서 가져오기 (필요하면 필드별 projection 조절 가능)
documents = collection.find({}, {"description": 1, "name": 1, "spec_table": 1})

for doc in documents:
    # 각 필드별 텍스트 추출
    texts = []
    for field in ["description", "name", "spec_table"]:
        val = doc.get(field)
        text = get_text_from_field(val)
        texts.append(text)
    combined_text = " ".join(texts)

    # 각 키워드별로 등장 횟수 집계 (대소문자 무시)
    for kw in keywords:
        word_count[kw] += len(re.findall(kw, combined_text, re.IGNORECASE))

# 결과 출력 - 등장 횟수 높은 순 정렬
for kw, cnt in word_count.most_common():
    print(f"{kw}: {cnt}회")


OEM: 40회
SA: 26회
Cherry: 18회
XDA: 16회
DSA: 3회
KAT: 0회
KAM: 0회
MT3: 0회


## 가격대

In [16]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
db = client['kikihi']  # 본인 DB명에 맞게 변경
collection = db['keyboard']

# 가격 구간 설정 (단위: 원)
price_bins = [
    (0, 50000),
    (50001, 100000),
    (100001, 200000),
    (200001, float('inf'))  # 20만원 초과
]

# 구간별 카운트 저장할 딕셔너리 초기화
price_counts = {
    "0~50,000": 0,
    "50,001~100,000": 0,
    "100,001~200,000": 0,
    "200,001 이상": 0
}

# 전체 문서 조회
for doc in collection.find({}, {"price": 1}):
    price = doc.get("price")

    if price is None:
        continue  # price 필드 없으면 건너뜀

    # price가 문자열이면 숫자로 변환 시도 (예: "120000원")
    if isinstance(price, str):
        price_str = price.replace(',', '').replace('원', '').strip()
        try:
            price = int(price_str)
        except ValueError:
            continue  # 변환 실패하면 무시

    # price가 숫자인지 체크
    if not isinstance(price, (int, float)):
        continue

    # 각 구간별 카운트 증가
    for i, (low, high) in enumerate(price_bins):
        if low <= price <= high:
            key = list(price_counts.keys())[i]
            price_counts[key] += 1
            break

# 결과 출력
for k, v in price_counts.items():
    print(f"{k}: {v}건")


0~50,000: 78건
50,001~100,000: 135건
100,001~200,000: 52건
200,001 이상: 38건


0~50,000: 145건
50,001~100,000: 80건
100,001~200,000: 40건
200,001 이상: 38건

## 경우의 수

In [9]:
from pymongo import MongoClient

MONGO_URI = "mongodb://localhost:27017/"
DB_NAME = "kikihi"
COLLECTION_NAME = "keyboard"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]
collection = db[COLLECTION_NAME]

# 검사할 키워드 정의
sizes = ["미니", "텐키리스", "풀배열"]  # description 필드
materials = ["알루미늄","ABS", "PBT", "OEM", "Cherry", "SA", "DSA", "XDA"]  # description, spec_table 필드
options_keys = ["적축", "갈축", "청축", "흑축", "자석축"]  # options 필드

# 가격대별 구간 정의 예 (0~50k, 50,001~100k, 100,001~200k, 200,001 이상)
# price 필드는 숫자형 또는 문자열형일 수 있으니 예시 처리 참고
price_ranges = {
    "0~50000": 0,
    "50001~100000": 0,
    "100001~200000": 0,
    "200001~": 0
}

# 개수 초기화 딕셔너리
size_counts = {size:0 for size in sizes}
material_counts = {mat:0 for mat in materials}
option_counts = {opt:0 for opt in options_keys}

# MongoDB 전체 문서 순회
for doc in collection.find():
    # 1. price 필드 파싱 및 카운트
    price = doc.get("price", "")
    price_num = None
    if isinstance(price, int) or isinstance(price, float):
        price_num = price
    elif isinstance(price, str):
        # 숫자만 추출 (예: "123,000원" -> 123000)
        import re
        nums = re.findall(r"\d+", price.replace(",", ""))
        if nums:
            price_num = int("".join(nums))
    # 가격대 카운트
    if price_num is not None:
        if price_num <= 50000:
            price_ranges["0~50000"] += 1
        elif price_num <= 100000:
            price_ranges["50001~100000"] += 1
        elif price_num <= 200000:
            price_ranges["100001~200000"] += 1
        else:
            price_ranges["200001~"] += 1

    # 2. size: description 내 단어 포함여부
    description = doc.get("description", "")
    if isinstance(description, list):
        desc_text = " ".join(map(str, description))
    else:
        desc_text = str(description)
    for size in sizes:
        if size in desc_text:
            size_counts[size] += 1

    # 3. material: description 또는 spec_table 내 존재여부
    # spec_table은 dict라고 가정하고, 모든 값 대상으로 검사
    spec_table = doc.get("spec_table", {})
    spec_values = " ".join([str(v) for v in spec_table.values()]) if isinstance(spec_table, dict) else ""

    for mat in materials:
        if mat in desc_text or mat in spec_values:
            material_counts[mat] += 1

    # 4. options 필드 배열 내 포함 여부
    options = doc.get("options", [])
    if isinstance(options, list):
        for opt_key in options_keys:
            # 옵션 리스트의 각 항목에 opt_key 포함 여부 검색 (대소문자 무시 가능)
            if any(opt_key in str(opt) for opt in options):
                option_counts[opt_key] += 1

# 출력 결과
print("=== 가격대별 개수 ===")
for pr, count in price_ranges.items():
    print(f"{pr}: {count}개")

print("\n=== 사이즈(description) 포함 개수 ===")
for size, count in size_counts.items():
    print(f"{size}: {count}개")

print("\n=== 재질(description, spec_table) 포함 개수 ===")
for mat, count in material_counts.items():
    print(f"{mat}: {count}개")

print("\n=== 옵션(options) 포함 개수 ===")
for opt, count in option_counts.items():
    print(f"{opt}: {count}개")


=== 가격대별 개수 ===
0~50000: 58개
50001~100000: 109개
100001~200000: 36개
200001~: 28개

=== 사이즈(description) 포함 개수 ===
미니: 37개
텐키리스: 65개
풀배열: 127개

=== 재질(description, spec_table) 포함 개수 ===
알루미늄: 8개
ABS: 72개
PBT: 135개
OEM: 0개
Cherry: 0개
SA: 14개
DSA: 0개
XDA: 0개

=== 옵션(options) 포함 개수 ===
적축: 77개
갈축: 77개
청축: 60개
흑축: 3개
자석축: 0개


In [10]:
from collections import Counter

combination_counts = Counter()

for doc in collection.find():
    # 가격 계산 (앞서와 같이)
    price = doc.get("price", "")
    price_num = None
    if isinstance(price, (int, float)):
        price_num = price
    elif isinstance(price, str):
        import re
        nums = re.findall(r"\d+", price.replace(",", ""))
        if nums:
            price_num = int("".join(nums))

    # 가격대 판별
    if price_num is None:
        price_range = "가격정보없음"
    elif price_num <= 50000:
        price_range = "0~50000"
    elif price_num <= 100000:
        price_range = "50001~100000"
    elif price_num <= 200000:
        price_range = "100001~200000"
    else:
        price_range = "200001~"

    # 사이즈 확인(description에서)
    description = doc.get("description", "")
    if isinstance(description, list):
        desc_text = " ".join(map(str, description)).lower()
    else:
        desc_text = str(description).lower()

    size_found = None
    for size in ["미니", "텐키리스", "풀배열"]:
        if size in desc_text:
            size_found = size
            break
    if not size_found:
        size_found = "사이즈정보없음"

    # 재질확인(description, spec_table, name)
    spec_table = doc.get("spec_table", {})
    spec_values = " ".join([str(v).lower() for v in spec_table.values()]) if isinstance(spec_table, dict) else ""
    name = doc.get("name", "").lower()

    material_found = None
    for mat in ["abs", "pbt", "oem", "cherry", "sa", "dsa", "xda"]:
        if mat in desc_text or mat in spec_values or mat in name:
            material_found = mat
            break
    if not material_found:
        material_found = "X"

    # 옵션확인(description, options)
    options = doc.get("options", [])
    options_lower = [str(opt).lower() for opt in options]

    option_found = None
    for opt_key in ["적축", "갈축", "청축", "흑축", "자석축","백축"]:
        if (opt_key in desc_text) or any(opt_key == opt for opt in options_lower):
            option_found = opt_key
            break
    if not option_found:
        option_found = "X"

    # 조합 키 생성
    key = (price_range, size_found, material_found, option_found)
    combination_counts[key] += 1

# 결과 출력 예시
for key, count in combination_counts.items():
    price_r, size_, material_, option_ = key
    print(f"가격대: {price_r}, 사이즈: {size_}, 재질: {material_}, 옵션: {option_} => {count}개")


가격대: 50001~100000, 사이즈: 풀배열, 재질: pbt, 옵션: X => 38개
가격대: 0~50000, 사이즈: 풀배열, 재질: X, 옵션: 적축 => 6개
가격대: 0~50000, 사이즈: 텐키리스, 재질: pbt, 옵션: X => 8개
가격대: 0~50000, 사이즈: 풀배열, 재질: abs, 옵션: 적축 => 19개
가격대: 50001~100000, 사이즈: 텐키리스, 재질: pbt, 옵션: X => 19개
가격대: 0~50000, 사이즈: 풀배열, 재질: pbt, 옵션: X => 4개
가격대: 200001~, 사이즈: 풀배열, 재질: pbt, 옵션: X => 4개
가격대: 200001~, 사이즈: 텐키리스, 재질: abs, 옵션: X => 1개
가격대: 200001~, 사이즈: 미니, 재질: pbt, 옵션: X => 4개
가격대: 0~50000, 사이즈: 풀배열, 재질: pbt, 옵션: 적축 => 2개
가격대: 0~50000, 사이즈: 텐키리스, 재질: abs, 옵션: 적축 => 4개
가격대: 50001~100000, 사이즈: 풀배열, 재질: abs, 옵션: 적축 => 5개
가격대: 100001~200000, 사이즈: 풀배열, 재질: abs, 옵션: X => 5개
가격대: 200001~, 사이즈: 풀배열, 재질: pbt, 옵션: 적축 => 2개
가격대: 100001~200000, 사이즈: 미니, 재질: X, 옵션: X => 1개
가격대: 100001~200000, 사이즈: 풀배열, 재질: abs, 옵션: 적축 => 3개
가격대: 200001~, 사이즈: 미니, 재질: abs, 옵션: X => 1개
가격대: 100001~200000, 사이즈: 텐키리스, 재질: pbt, 옵션: X => 3개
가격대: 0~50000, 사이즈: 사이즈정보없음, 재질: X, 옵션: X => 1개
가격대: 0~50000, 사이즈: 풀배열, 재질: pbt, 옵션: 백축 => 1개
가격대: 50001~100000, 사이즈: 미니, 재질: pbt, 옵션: X => 11개


In [11]:
import pandas as pd
from collections import Counter

# --- (이전에 MongoDB에서 집계된 결과 모양) ---
# combination_counts 예시 (실제에선 이전 코드에서 집계된 결과를 사용하세요)
# combination_counts = Counter({
#     ("0~50000", "미니", "abs", "적축"): 5,
#     ("50001~100000", "텐키리스", "cherry", "청축"): 3,
#     # ...
# })

# 아래 부분만 복사해서 실행하세요 (combination_counts 는 반드시 기존 집계 결과 변수명과 동일해야 합니다)

# combination_counts 변수가 없다면, 아래 예시용 임시 데이터를 넣을 수 있습니다:
# 아래 주석 해제하면 테스트 가능
# combination_counts = Counter({
#     ("0~50000", "미니", "abs", "적축"): 5,
#     ("50001~100000", "텐키리스", "cherry", "청축"): 3,
#     ("100001~200000", "풀배열", "pbt", "흑축"): 1,
#     ("200001~", "풀배열", "oem", "자석축"): 2,
# })

# 데이터프레임으로 변환
data = []
for key, count in combination_counts.items():
    price_range, size, material, option = key
    data.append({
        "가격대": price_range,
        "사이즈": size,
        "재질": material,
        "옵션": option,
        "개수": count
    })

df = pd.DataFrame(data)

# 보기 좋게 정렬
df = df.sort_values(by=["가격대", "사이즈", "재질", "옵션"]).reset_index(drop=True)

# 콘솔에 표 출력
print(df)


              가격대      사이즈   재질  옵션  개수
0         0~50000       미니  abs  갈축   1
1         0~50000       미니  pbt   X   2
2         0~50000  사이즈정보없음    X   X   1
3         0~50000     텐키리스    X  적축   1
4         0~50000     텐키리스  abs  갈축   1
5         0~50000     텐키리스  abs  적축   4
6         0~50000     텐키리스  pbt   X   8
7         0~50000     텐키리스  pbt  갈축   1
8         0~50000     텐키리스  pbt  백축   1
9         0~50000     텐키리스  pbt  적축   1
10        0~50000      풀배열    X   X   1
11        0~50000      풀배열    X  적축   6
12        0~50000      풀배열  abs   X   3
13        0~50000      풀배열  abs  적축  19
14        0~50000      풀배열  abs  청축   1
15        0~50000      풀배열  pbt   X   4
16        0~50000      풀배열  pbt  백축   1
17        0~50000      풀배열  pbt  적축   2
18  100001~200000       미니    X   X   1
19  100001~200000       미니  abs  적축   1
20  100001~200000       미니  pbt   X   5
21  100001~200000       미니  pbt  적축   1
22  100001~200000  사이즈정보없음  pbt  적축   1
23  100001~200000     텐키리스    X  적축   1


In [12]:
import pandas as pd

# 모든 행을 출력하도록 설정
pd.set_option('display.max_rows', None)

# (이미 존재하는 df를 출력)
print(df)

              가격대      사이즈   재질  옵션  개수
0         0~50000       미니  abs  갈축   1
1         0~50000       미니  pbt   X   2
2         0~50000  사이즈정보없음    X   X   1
3         0~50000     텐키리스    X  적축   1
4         0~50000     텐키리스  abs  갈축   1
5         0~50000     텐키리스  abs  적축   4
6         0~50000     텐키리스  pbt   X   8
7         0~50000     텐키리스  pbt  갈축   1
8         0~50000     텐키리스  pbt  백축   1
9         0~50000     텐키리스  pbt  적축   1
10        0~50000      풀배열    X   X   1
11        0~50000      풀배열    X  적축   6
12        0~50000      풀배열  abs   X   3
13        0~50000      풀배열  abs  적축  19
14        0~50000      풀배열  abs  청축   1
15        0~50000      풀배열  pbt   X   4
16        0~50000      풀배열  pbt  백축   1
17        0~50000      풀배열  pbt  적축   2
18  100001~200000       미니    X   X   1
19  100001~200000       미니  abs  적축   1
20  100001~200000       미니  pbt   X   5
21  100001~200000       미니  pbt  적축   1
22  100001~200000  사이즈정보없음  pbt  적축   1
23  100001~200000     텐키리스    X  적축   1


## 한번만 다시

In [13]:
import re
from collections import Counter
import pandas as pd
from pymongo import MongoClient

# MongoDB 연결
MONGO_URI = "mongodb://localhost:27017/"
DB_NAME = "kikihi"
COLLECTION_NAME = "keyboard"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]
collection = db[COLLECTION_NAME]

# 검색 키워드 리스트 정의
sizes = ["미니", "텐키리스", "풀배열"]  # description 내
materials = ["알루미늄", "abs", "pbt", "oem", "cherry", "sa", "dsa", "xda"]  # lower case 변환해서 비교
options_keys = ["적축", "갈축", "청축", "흑축", "자석축"]  # options 필드 및 description 내

combination_counts = Counter()

for doc in collection.find():
    # 1. 가격 숫자 추출 및 구간 분류
    price = doc.get("price", "")
    price_num = None
    if isinstance(price, (int, float)):
        price_num = price
    elif isinstance(price, str):
        nums = re.findall(r"\d+", price.replace(",", ""))
        if nums:
            price_num = int("".join(nums))

    if price_num is None:
        price_range = "가격정보없음"
    elif price_num <= 50000:
        price_range = "0~50000"
    elif price_num <= 100000:
        price_range = "50001~100000"
    elif price_num <= 200000:
        price_range = "100001~200000"
    else:
        price_range = "200001~"

    # 2. 사이즈 찾기 (description)
    description = doc.get("description", "")
    if isinstance(description, list):
        desc_text = " ".join(map(str, description)).lower()
    else:
        desc_text = str(description).lower()

    size_found = None
    for size in sizes:
        if size in desc_text:
            size_found = size
            break
    if not size_found:
        size_found = "사이즈정보없음"

    # 3. 재질 찾기 (description, spec_table, name)
    spec_table = doc.get("spec_table", {})
    spec_values = ""
    if isinstance(spec_table, dict):
        spec_values = " ".join([str(v).lower() for v in spec_table.values()])

    name = doc.get("name", "").lower()

    material_found = None
    for mat in materials:
        mat_lower = mat.lower()
        # '알루미늄'은 한글 그대로 검색, 나머지는 소문자로 변환 후 검색
        if mat_lower == "알루미늄":
            check_texts = [desc_text, spec_values, name]
            if any(mat in text for text in check_texts):
                material_found = mat
                break
        else:
            if (mat_lower in desc_text) or (mat_lower in spec_values) or (mat_lower in name):
                material_found = mat
                break
    if not material_found:
        material_found = "재질정보없음"

    # 4. 옵션 찾기 (options 리스트 + description)
    options = doc.get("options", [])
    options_lower = [str(opt).lower() for opt in options]

    option_found = None
    for opt_key in options_keys:
        opt_key_lower = opt_key.lower()
        if (opt_key_lower in desc_text) or any(opt_key_lower == opt for opt in options_lower):
            option_found = opt_key
            break
    if not option_found:
        option_found = "옵션정보없음"

    # 5. 조합 키 생성 및 카운트 증가
    key = (price_range, size_found, material_found, option_found)
    combination_counts[key] += 1

# 6. pandas DataFrame으로 변환
data = []
for key, count in combination_counts.items():
    price_range, size, material, option = key
    data.append({
        "가격대": price_range,
        "사이즈": size,
        "재질": material,
        "옵션": option,
        "개수": count
    })

df = pd.DataFrame(data)

# 7. 보기 좋게 정렬 (가격대 순서 맞춤)
price_order = ["가격정보없음", "0~50000", "50001~100000", "100001~200000", "200001~"]
df['가격대'] = pd.Categorical(df['가격대'], categories=price_order, ordered=True)
df = df.sort_values(by=["가격대", "사이즈", "재질", "옵션"]).reset_index(drop=True)

# 8. 결과 출력
print(df)


              가격대      사이즈      재질      옵션  개수
0         0~50000       미니     abs      갈축   1
1         0~50000       미니     pbt  옵션정보없음   2
2         0~50000  사이즈정보없음  재질정보없음  옵션정보없음   1
3         0~50000     텐키리스     abs      갈축   1
4         0~50000     텐키리스     abs      적축   4
5         0~50000     텐키리스     pbt      갈축   1
6         0~50000     텐키리스     pbt  옵션정보없음   9
7         0~50000     텐키리스     pbt      적축   1
8         0~50000     텐키리스  재질정보없음      적축   1
9         0~50000      풀배열     abs  옵션정보없음   3
10        0~50000      풀배열     abs      적축  19
11        0~50000      풀배열     abs      청축   1
12        0~50000      풀배열     pbt  옵션정보없음   5
13        0~50000      풀배열     pbt      적축   2
14        0~50000      풀배열  재질정보없음  옵션정보없음   1
15        0~50000      풀배열  재질정보없음      적축   6
16   50001~100000       미니     abs      적축   1
17   50001~100000       미니     pbt  옵션정보없음   8
18   50001~100000       미니     pbt      적축   1
19   50001~100000       미니    알루미늄  옵션정보없음   3
20   50001~10